# Valoricore × LangChain

**Deterministic vector memory with cryptographic audit trails — natively in LangChain.**

| Section | What you'll see |
|---|---|
| 1–3 | Install, imports, one-line init |
| 4–5 | Add texts and documents |
| 6–8 | Search variants |
| 9 | **Index comparison — BruteForce vs HNSW** |
| 10 | Tag-filtered search (tenant isolation) |
| 11 | As retriever + RAG chain |
| 12 | From texts factory |
| 13–14 | State hash + Snapshot/restore |

[![GitHub](https://img.shields.io/badge/GitHub-Valori--Kernel-6c47ff?logo=github)](https://github.com/varshith-Git/Valori-Kernel)

## 1 · Install

In [ ]:
%%capture
!pip install "valoricore[langchain,local]"
# langchain  → langchain-core + langchain
# local      → sentence-transformers (offline embeddings, no API key needed)

## 2 · Imports & embedder

In [ ]:
import time, statistics
from valoricore.integrations import ValoricoreLangChain
from valoricore.embeddings   import SentenceTransformerEmbedder

class LocalEmbedder:
    """Wraps SentenceTransformerEmbedder to satisfy LangChain's Embeddings interface."""
    def __init__(self, model="all-MiniLM-L6-v2"):
        self._e = SentenceTransformerEmbedder(model)
    def embed_documents(self, texts):
        return self._e.embed_batch(texts)
    def embed_query(self, text):
        return self._e.embed(text)

embedding = LocalEmbedder()
print(f"Embedding dim: {len(embedding.embed_query('test'))}")

## 3 · One-line initialization

In [ ]:
store = ValoricoreLangChain(
    path       = "/tmp/valori_langchain_demo",
    embedding  = embedding,
    index_kind = "bruteforce",   # changed per-section below
)
print(f"Store initialized ✓  |  hash: {store.get_state_hash()[:16]}...")

## 4 · Add texts

In [ ]:
texts = [
    "Valoricore is a deterministic vector database built in Rust.",
    "Q16.16 fixed-point arithmetic produces identical results on x86, ARM, and RISC-V.",
    "Every insert is backed by a BLAKE3 Merkle proof.",
    "The event log is append-only and fsynced before any live state change.",
    "LangChain users can swap FAISS for Valoricore with one line of code.",
    "LlamaIndex users can use ValoricoreLlamaIndex as a drop-in vector store.",
    "Crash recovery is mathematically proven, not just claimed.",
    "Leader-follower replication uses tokio::sync::watch for race-free coordination.",
]
metadatas = [
    {"source": "overview",     "topic": "intro"},
    {"source": "architecture", "topic": "determinism"},
    {"source": "architecture", "topic": "crypto"},
    {"source": "architecture", "topic": "durability"},
    {"source": "integrations", "topic": "langchain"},
    {"source": "integrations", "topic": "llamaindex"},
    {"source": "overview",     "topic": "crash-recovery"},
    {"source": "architecture", "topic": "replication"},
]
ids = store.add_texts(texts, metadatas)
print(f"Inserted {len(ids)} records | hash: {store.get_state_hash()[:16]}...")

## 5 · Add LangChain Documents

In [ ]:
from langchain_core.documents import Document

documents = [
    Document(page_content="The HNSW index enables sub-millisecond search at millions of records.",
             metadata={"source": "architecture", "topic": "hnsw"}),
    Document(page_content="The IVF index uses deterministic k-means with FNV-1a seeding.",
             metadata={"source": "architecture", "topic": "ivf"}),
]
ids = store.add_documents(documents)
print(f"Inserted {len(ids)} documents, IDs: {ids}")

## 6 · Similarity search

In [ ]:
results = store.similarity_search("How does crash recovery work?", k=3)
for i, doc in enumerate(results, 1):
    print(f"{i}. [{doc.metadata.get('topic')}] {doc.page_content}")

## 7 · Similarity search with distance scores
Score is the raw Q16.16² L2 distance — **lower = more similar**.

In [ ]:
pairs = store.similarity_search_with_score("deterministic arithmetic", k=4)
for doc, score in pairs:
    print(f"  score={score:.1f}  →  {doc.page_content[:65]}")

## 8 · Search by pre-computed vector

In [ ]:
vec  = embedding.embed_query("BLAKE3 Merkle proof")
docs = store.similarity_search_by_vector(vec, k=2)
for doc in docs:
    print("→", doc.page_content)

---
## 9 · Index Comparison — BruteForce vs HNSW

### What each index is

| Index | Algorithm | Recall | Latency | Best for |
|---|---|---|---|---|
| `bruteforce` | Full scan every query | **100%** exact | O(n) linear | < 50K vectors, compliance, testing |
| `hnsw` | Multi-layer proximity graph | ~95%+ ANN | O(log n) | > 100K vectors, production throughput |

**HNSW parameters in Valoricore's Rust kernel:**
- `M = 16` — bidirectional links per node (higher = better recall, more RAM)
- `M_MAX = 32` — max links at layer 0
- `EF_CONSTRUCTION = 64` — candidate pool size during graph construction
- **All distance math is Q16.16 fixed-point** — bit-identical results on every architecture

> **IVF note:** IVF (cluster-based index) is on the Valoricore roadmap. When available
> it will sit between BruteForce and HNSW: faster than BruteForce, more recall-predictable
> than HNSW. Use `bruteforce` or `hnsw` for current deployments.

In [ ]:
# Build two stores — same data, different indexes
bf_store = ValoricoreLangChain(
    path      = "/tmp/valori_lc_bruteforce",
    embedding = embedding,
    index_kind = "bruteforce",
)
hnsw_store = ValoricoreLangChain(
    path      = "/tmp/valori_lc_hnsw",
    embedding = embedding,
    index_kind = "hnsw",
)

# Shared corpus — realistic mix of topics
corpus = [
    "Valoricore stores vectors with cryptographic proofs.",
    "HNSW graph construction uses fixed-point distance math.",
    "BruteForce scans every record — zero approximation.",
    "The BLAKE3 state root changes after every write.",
    "Event sourcing ensures crash safety before live apply.",
    "Tag filtering is applied at the kernel level, not query time.",
    "Snapshot and restore operations are bit-exact across machines.",
    "Knowledge graphs link document nodes to chunk nodes.",
    "Q16.16 fixed-point gives identical results on x86 and ARM.",
    "Leader-follower replication is coordinated via watch channels.",
]

bf_store.add_texts(corpus)
hnsw_store.add_texts(corpus)

print(f"BruteForce store — {len(corpus)} records, hash: {bf_store.get_state_hash()[:16]}...")
print(f"HNSW store       — {len(corpus)} records, hash: {hnsw_store.get_state_hash()[:16]}...")

In [ ]:
# Benchmark: latency and recall across multiple queries
test_queries = [
    "cryptographic proof and audit",
    "approximate nearest neighbor graph",
    "crash recovery and durability",
    "tenant isolation and tag filtering",
    "fixed-point deterministic arithmetic",
]
K = 3

bf_latencies, hnsw_latencies, recalls = [], [], []

for q in test_queries:
    # BruteForce — ground truth
    t0 = time.perf_counter()
    gt = bf_store.similarity_search(q, k=K)
    bf_latencies.append((time.perf_counter() - t0) * 1000)

    # HNSW — approximate
    t0 = time.perf_counter()
    ap = hnsw_store.similarity_search(q, k=K)
    hnsw_latencies.append((time.perf_counter() - t0) * 1000)

    gt_set = {d.page_content for d in gt}
    ap_set = {d.page_content for d in ap}
    recalls.append(len(gt_set & ap_set) / K)

print("── LangChain Index Benchmark ────────────────────────────────")
print(f"  {'Index':<12} {'Avg (ms)':>10} {'Max (ms)':>10}")
print(f"  {'bruteforce':<12} {statistics.mean(bf_latencies):>10.3f} {max(bf_latencies):>10.3f}")
print(f"  {'hnsw':<12} {statistics.mean(hnsw_latencies):>10.3f} {max(hnsw_latencies):>10.3f}")
print(f"\n  HNSW recall vs BruteForce ground truth: {statistics.mean(recalls)*100:.1f}%")

print("\n── Per-query recall ──────────────────────────────────────────")
for q, r, bms, hms in zip(test_queries, recalls, bf_latencies, hnsw_latencies):
    bar = '█' * int(r * 10) + '░' * (10 - int(r * 10))
    print(f"  {bar}  {r*100:5.1f}%  BF={bms:.2f}ms  HNSW={hms:.2f}ms  '{q}'")

In [ ]:
# Side-by-side result comparison for one query
demo_q = "deterministic fixed-point arithmetic"

bf_results   = bf_store.similarity_search_with_score(demo_q, k=3)
hnsw_results = hnsw_store.similarity_search_with_score(demo_q, k=3)

print(f"Query: '{demo_q}'\n")
print(f"{'BruteForce (exact)':^55}  {'HNSW (approximate)':^55}")
print("-" * 55 + "  " + "-" * 55)

for (bd, bs), (hd, hs) in zip(bf_results, hnsw_results):
    bf_line  = f"{bs:>10.0f}  {bd.page_content[:40]}"
    hn_line  = f"{hs:>10.0f}  {hd.page_content[:40]}"
    match    = "✓" if bd.page_content == hd.page_content else "≠"
    print(f"{bf_line:<55}  {hn_line:<55}  {match}")

### When to choose which index

```
Dataset size?
  < 50K vectors  →  bruteforce (exact, zero config, zero recall loss)
  > 100K vectors →  hnsw       (sub-ms search, ~95%+ recall)
  50–100K        →  start with bruteforce, migrate to hnsw when latency matters

Use case?
  Compliance / audit trail  →  bruteforce (100% recall is provable)
  Production RAG            →  hnsw       (throughput & latency)
  Testing / CI              →  bruteforce (deterministic, predictable)
```

---
## 10 · Tag-Filtered Search (Tenant Isolation)

In [ ]:
tagged = ValoricoreLangChain(path="/tmp/valori_lc_tagged", embedding=embedding)

tagged.add_texts(
    texts     = ["Tenant A confidential report", "Tenant A internal memo"],
    metadatas = [{"tenant": "A"}, {"tenant": "A"}],
    tags      = [1, 1],
)
tagged.add_texts(
    texts     = ["Tenant B confidential report", "Tenant B internal memo"],
    metadatas = [{"tenant": "B"}, {"tenant": "B"}],
    tags      = [2, 2],
)

docs_a = tagged.similarity_search("confidential report", k=5, filter_tag=1)
docs_b = tagged.similarity_search("confidential report", k=5, filter_tag=2)

print(f"Tenant A results: {len(docs_a)} | Tenant B results: {len(docs_b)}")
a_ids = {d.page_content for d in docs_a}
b_ids = {d.page_content for d in docs_b}
print(f"Cross-tenant leakage: {a_ids & b_ids} (empty ✓)")

## 11 · Use as a LangChain retriever

In [ ]:
retriever = store.as_retriever(k=3)
docs = retriever.get_relevant_documents("fixed-point arithmetic")
print(f"Retriever returned {len(docs)} docs:")
for doc in docs:
    print(f"  · {doc.page_content[:75]}")

In [ ]:
# ── Full RAG with OpenAI (requires OPENAI_API_KEY) ───────────────────────────
# from langchain.chains import RetrievalQA
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
#
# openai_store = ValoricoreLangChain(
#     path="/tmp/valori_openai", embedding=OpenAIEmbeddings()
# )
# openai_store.add_texts(texts, metadatas)
# chain  = RetrievalQA.from_chain_type(
#     llm=ChatOpenAI(model="gpt-4o-mini"),
#     retriever=openai_store.as_retriever(k=4),
# )
# answer = chain.run("How does Valoricore prove crash recovery?")
# print(answer)

## 12 · From texts (factory pattern)

In [ ]:
fresh = ValoricoreLangChain.from_texts(
    texts=["factory doc 1", "factory doc 2"],
    embedding=embedding,
    path="/tmp/valori_lc_factory",
)
print(f"from_texts: {len(fresh.similarity_search('factory', k=2))} result(s)")

## 13 · Cryptographic state hash

In [ ]:
h1 = store.get_state_hash()
store.add_texts(["post-hash document"])
h2 = store.get_state_hash()
print(f"Before: {h1}")
print(f"After : {h2}")
print(f"Differ: {h1 != h2} ✓")

## 14 · Snapshot & bit-exact restore

In [ ]:
h_before = store.get_state_hash()
snap     = store.snapshot()

restored = ValoricoreLangChain(path="/tmp/valori_lc_restored", embedding=embedding)
restored.restore(snap)

h_after = restored.get_state_hash()
print(f"Snapshot : {len(snap):,} bytes")
print(f"Before   : {h_before}")
print(f"After    : {h_after}")
print(f"Bit-exact: {h_before == h_after} ✓")

---
## Summary

| Feature | API |
|---|---|
| Init (local) | `ValoricoreLangChain(path=..., embedding=..., index_kind="bruteforce\|hnsw")` |
| Init (remote) | `ValoricoreLangChain(remote="http://...", embedding=...)` |
| Add texts | `store.add_texts(texts, metadatas, tags=[1,2,...])` |
| Search | `store.similarity_search(query, k=5)` |
| Search + score | `store.similarity_search_with_score(query, k=5)` |
| Tag filter | `store.similarity_search(query, filter_tag=1)` |
| As retriever | `store.as_retriever(k=5)` |
| State proof | `store.get_state_hash()` — 64-char BLAKE3 |
| Snapshot | `store.snapshot()` / `store.restore(snap)` |

**Other notebooks:**
- [End-to-End Demo](https://colab.research.google.com/drive/1QO1yQMQoGbp9fwrb00KVKTq5bYVGXgJv)
- [LlamaIndex Integration](https://colab.research.google.com/drive/1Q72ANZxBm1fthNpgVW-FftS8sZz6uCr3)
- [GitHub](https://github.com/varshith-Git/Valori-Kernel)